# ДЗ 7. Часові ряди та напівструктуровані дані: NYC Taxi time-series features, JSONB event-log, GIN-індекси

**Дані (варіант A):** офіційний NYC TLC Trip Record Data, файл `yellow_tripdata_2024-03.parquet` (березень 2024, 3.58 млн поїздок, 60 МБ) з https://d37ci6vzurychx.cloudfront.net/trip-data/. Файл завантажується кодом у ноутбуці; у репозиторій не додається.

**Вибірка:** замість `head(500_000)` (перші 500 тис. рядків — це лише перші ~4 дні місяця, і 30-денні лаги були б порожніми) береться кожен 7-й рядок файлу (`iloc[::7]`, ≈ 512 тис. поїздок), що рівномірно покриває всі 31 день березня. Далі базове очищення за умовою (`0 < fare < 500`, `0 < distance < 100`, `PULocationID` не NULL).

**Середовище:** у Colab PostgreSQL 16 через `pgserver`; локально перевірено на Docker `postgres:16` через `HW7_PG_URI`. Службові таблиці з префіксом `t7_hw_`, усі блоки починаються з `DROP TABLE IF EXISTS`.

In [ ]:
# ---------- Завдання 1. Setup ----------
import os, sys, subprocess, importlib

def ensure(pkg, module=None):
    try:
        importlib.import_module(module or pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

for pkg, mod in [("psycopg2-binary", "psycopg2"), ("sqlalchemy", "sqlalchemy"), ("pandas", "pandas"), ("pyarrow", "pyarrow"), ("requests", "requests")]:
    ensure(pkg, mod)

from urllib.request import urlretrieve
import pandas as pd
from sqlalchemy import create_engine, text

PG_URI = os.environ.get("HW7_PG_URI")
if PG_URI:
    print("Зовнішній PostgreSQL:", PG_URI.split("@")[-1])
else:
    ensure("pgserver")
    import pgserver
    pg = pgserver.get_server("/tmp/hw7_pg", cleanup_mode="stop")
    PG_URI = pg.get_uri()
    print("pgserver запущено:", PG_URI.split("@")[-1])

engine = create_engine(PG_URI, future=True)
with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())

def q(sql, **params):
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn, params=params)

def run(sql):
    with engine.begin() as conn:
        res = conn.execute(text(sql))
        if res.returns_rows:
            return pd.DataFrame(res.fetchall(), columns=list(res.keys()))

def explain(sql):
    '''EXPLAIN (ANALYZE, BUFFERS) -> друкує план як текст'''
    print("\n".join(q("EXPLAIN (ANALYZE, BUFFERS) " + sql).iloc[:, 0]))

pd.set_option("display.width", 180); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 80)

In [ ]:
URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-03.parquet"
DATA_DIR = "/content" if os.path.isdir("/content") else "."
parquet_path = os.path.join(DATA_DIR, "yellow_tripdata_2024-03.parquet")
if not os.path.exists(parquet_path):
    urlretrieve(URL, parquet_path)
print("Parquet:", parquet_path, f"({os.path.getsize(parquet_path) / 1e6:.0f} МБ)")

columns = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID",
           "trip_distance", "fare_amount", "passenger_count"]
df_all = pd.read_parquet(parquet_path, columns=columns)
print("Усього поїздок у файлі:", df_all.shape)

df = df_all.iloc[::7].copy()                      # кожен 7-й рядок - рівномірно по всьому місяцю
del df_all

# Базове очищення перед завантаженням
df = df[(df["fare_amount"] > 0) & (df["fare_amount"] < 500)]
df = df[(df["trip_distance"] > 0) & (df["trip_distance"] < 100)]
df = df[df["PULocationID"].notna()]
# у файлі трапляються поодинокі поїздки з іншими місяцями - лишаємо лише березень 2024
df = df[(df["tpep_pickup_datetime"] >= "2024-03-01") & (df["tpep_pickup_datetime"] < "2024-04-01")]
print("Після вибірки й очищення:", df.shape)

df.to_sql("nyc_taxi_raw", engine, if_exists="replace", index=False, chunksize=10_000, method="multi")
q("SELECT COUNT(*) AS n_rows, MIN(tpep_pickup_datetime) AS first_pickup, MAX(tpep_pickup_datetime) AS last_pickup FROM nyc_taxi_raw")

In [ ]:
run('''
DROP TABLE IF EXISTS t7_hw_taxi_daily;
CREATE TABLE t7_hw_taxi_daily AS
SELECT
    "PULocationID"::INTEGER                                      AS pickup_zone_id,
    DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE     AS ts_day,
    COUNT(*)::INTEGER                                            AS trip_count,
    AVG(fare_amount)::NUMERIC(10, 2)                             AS avg_fare,
    SUM(trip_distance)::NUMERIC(12, 2)                           AS total_distance
FROM nyc_taxi_raw
WHERE fare_amount > 0 AND fare_amount < 500
  AND trip_distance > 0 AND trip_distance < 100
  AND "PULocationID" IS NOT NULL
GROUP BY "PULocationID"::INTEGER, DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE;

ALTER TABLE t7_hw_taxi_daily ADD PRIMARY KEY (pickup_zone_id, ts_day);
''')
print(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT pickup_zone_id) AS n_zones, COUNT(DISTINCT ts_day) AS n_days FROM t7_hw_taxi_daily").to_string(index=False))
q("SELECT pickup_zone_id, COUNT(*) AS n_days FROM t7_hw_taxi_daily GROUP BY pickup_zone_id ORDER BY pickup_zone_id LIMIT 10")

### Production-style typed-таблиця з `TIMESTAMPTZ`

Сирий `tpep_pickup_datetime` — локальний час Нью-Йорка без зсуву. `AT TIME ZONE 'America/New_York'` перетворює його на `TIMESTAMPTZ` (момент в UTC). Для березня 2024 це важливо ще й тому, що 10 березня о 02:00 відбувся перехід на літній час: година 02:00–03:00 локально не існує, і без явної зони такі моменти інтерпретуються неоднозначно.

In [ ]:
run('''
DROP TABLE IF EXISTS t7_hw_taxi_trips_typed;
CREATE TABLE t7_hw_taxi_trips_typed AS
SELECT
    "PULocationID"::INTEGER                                            AS pickup_zone_id,
    "DOLocationID"::INTEGER                                            AS dropoff_zone_id,
    (tpep_pickup_datetime::TIMESTAMP AT TIME ZONE 'America/New_York')  AS pickup_at,
    (tpep_dropoff_datetime::TIMESTAMP AT TIME ZONE 'America/New_York') AS dropoff_at,
    trip_distance::NUMERIC(10, 2)                                      AS trip_distance,
    fare_amount::NUMERIC(10, 2)                                        AS fare_amount,
    passenger_count::INTEGER                                           AS passenger_count
FROM nyc_taxi_raw
WHERE fare_amount > 0 AND fare_amount < 500 AND trip_distance > 0 AND trip_distance < 100;
''')
q("SELECT pickup_at, pg_typeof(pickup_at) AS pickup_type, pickup_at AT TIME ZONE 'America/New_York' AS local_ny FROM t7_hw_taxi_trips_typed ORDER BY pickup_at LIMIT 5")

**Результат setup.** У файлі 3 582 628 поїздок; після вибірки кожного 7-го рядка та очищення — 491 308 поїздок з 1 по 31 березня 2024. Daily-агрегація `t7_hw_taxi_daily` дає 5 415 рядків: 248 pickup-зон × до 31 дня. Зони нерівномірні: центральні (4, 7, 10, 12) мають усі 31 день, околичні (1, 6, 8) — 2–8 днів, бо у вибірці кожного 7-го рядка туди потрапили лише поодинокі поїздки. У typed-таблиці `pickup_at` має тип `timestamp with time zone`: локальна північ 1 березня стала `05:00+00` (UTC−5 до переходу на літній час).

## Завдання 2. Time-series ML features

Усі вікна — з `PARTITION BY pickup_zone_id`: без нього `LAG` на першому дні зони взяв би значення з останнього дня попередньої зони, і модель отримала б історію чужої сутності.

### Feature 1. Lag features: 1d, 7d, 30d

In [ ]:
q('''
SELECT pickup_zone_id, ts_day, trip_count,
       LAG(trip_count, 1)  OVER w AS trips_lag_1d,
       LAG(trip_count, 7)  OVER w AS trips_lag_7d,
       LAG(trip_count, 30) OVER w AS trips_lag_30d
FROM t7_hw_taxi_daily
WINDOW w AS (PARTITION BY pickup_zone_id ORDER BY ts_day)
ORDER BY pickup_zone_id, ts_day
LIMIT 20
''')

### Feature 2. Rolling 7-day average fare (`INTERVAL '6 days' PRECEDING` = 7 календарних днів включно з поточним)

In [ ]:
q('''
SELECT pickup_zone_id, ts_day, avg_fare,
       AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
       COUNT(*)      OVER w7 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w7 AS (PARTITION BY pickup_zone_id ORDER BY ts_day::TIMESTAMP
              RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW)
ORDER BY pickup_zone_id, ts_day
LIMIT 20
''')

### Feature 3. Rolling 30-day fare volatility

In [ ]:
q('''
SELECT pickup_zone_id, ts_day, avg_fare,
       STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,
       COUNT(*)              OVER w30 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w30 AS (PARTITION BY pickup_zone_id ORDER BY ts_day::TIMESTAMP
               RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW)
ORDER BY pickup_zone_id, ts_day
LIMIT 20
''')

### Feature 4. Expanding cumulative trips

In [ ]:
q('''
SELECT pickup_zone_id, ts_day, trip_count,
       SUM(trip_count) OVER we AS cumulative_trips,
       AVG(trip_count) OVER we AS expanding_avg_trips
FROM t7_hw_taxi_daily
WINDOW we AS (PARTITION BY pickup_zone_id ORDER BY ts_day
              ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
ORDER BY pickup_zone_id, ts_day
LIMIT 20
''')

### Combined feature view

In [ ]:
run('''
CREATE OR REPLACE VIEW v_t7_hw_taxi_features AS
WITH features_basic AS (
    SELECT
        pickup_zone_id, ts_day, trip_count, avg_fare, total_distance,
        LAG(trip_count, 1)  OVER w_pid AS trips_lag_1d,
        LAG(trip_count, 7)  OVER w_pid AS trips_lag_7d,
        LAG(trip_count, 30) OVER w_pid AS trips_lag_30d,
        AVG(avg_fare)         OVER w7  AS rolling_avg_fare_7d,
        STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,
        SUM(trip_count)       OVER we  AS cumulative_trips,
        AVG(trip_count)       OVER we  AS expanding_avg_trips
    FROM t7_hw_taxi_daily
    WINDOW
        w_pid AS (PARTITION BY pickup_zone_id ORDER BY ts_day),
        w7    AS (PARTITION BY pickup_zone_id ORDER BY ts_day::TIMESTAMP
                  RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW),
        w30   AS (PARTITION BY pickup_zone_id ORDER BY ts_day::TIMESTAMP
                  RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW),
        we    AS (PARTITION BY pickup_zone_id ORDER BY ts_day
                  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
)
SELECT * FROM features_basic
WHERE trips_lag_30d IS NOT NULL;
''')
print(q("SELECT COUNT(*) AS rows_with_full_history, COUNT(DISTINCT pickup_zone_id) AS zones, MIN(ts_day) AS first_day, MAX(ts_day) AS last_day FROM v_t7_hw_taxi_features").to_string(index=False))
q("SELECT * FROM v_t7_hw_taxi_features ORDER BY trip_count DESC, pickup_zone_id LIMIT 20")

**Що показують ознаки.** Усі чотири вікна мають `PARTITION BY pickup_zone_id`, тому на першому дні кожної зони `trips_lag_1d` — NULL, а не значення сусідньої зони. На розріджених зонах (1, 3) видно різницю між `ROWS` і `RANGE`: `LAG(trip_count, 1)` бере **попередній рядок** незалежно від розриву в датах (для зони 1 «lag 1d» на 11 березня — це 8 березня), тоді як `RANGE BETWEEN INTERVAL '6 days' PRECEDING` рахує **календарні** дні: `days_in_window` для зони 1 коливається від 1 до 5, і 28 березня вікно починається заново (попередній день 18 березня випав за межі 7 днів). `STDDEV_SAMP` у 30-денному вікні на першому рядку — NULL (один елемент), далі зростає зі стабілізацією. Expanding-ознаки монотонні. Для 30-денного лагу повна історія є лише у 90 зон на 31 березня (один місяць даних), саме їх і лишає view `WHERE trips_lag_30d IS NOT NULL`; лідери — зона 132 (JFK, 795 поїздок, середній тариф 64.82) і 138 (LaGuardia). Для production-датасету потрібно завантажити кілька місяців, інакше ознака 30d майже порожня.

## Завдання 3. JSONB-таблиця для подій

5 типів подій із різною структурою payload: `page_view`, `click`, `add_to_cart`, `checkout`, `recommendation_shown`.

In [ ]:
run('''
DROP TABLE IF EXISTS t7_hw_events;
CREATE TABLE t7_hw_events (
    event_id    INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    user_id     INTEGER NOT NULL,
    event_ts    TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    event_type  TEXT NOT NULL,
    payload     JSONB NOT NULL
);

INSERT INTO t7_hw_events (user_id, event_ts, event_type, payload)
SELECT
    (n % 1000) + 1 AS user_id,
    TIMESTAMPTZ '2024-04-01 00:00:00+00' - ((n % 60) || ' days')::INTERVAL AS event_ts,
    CASE (n % 5) WHEN 0 THEN 'page_view' WHEN 1 THEN 'click' WHEN 2 THEN 'add_to_cart'
                 WHEN 3 THEN 'checkout' ELSE 'recommendation_shown' END AS event_type,
    CASE (n % 5)
        WHEN 0 THEN jsonb_build_object(
            'url', '/product/' || ((n % 200) + 1),
            'referrer', (ARRAY['google','direct','email','fb'])[(n % 4) + 1],
            'device', (ARRAY['mobile','desktop','tablet'])[(n % 3) + 1])
        WHEN 1 THEN jsonb_build_object(
            'element_id', 'btn_' || ((n % 50) + 1),
            'page_url', '/category/' || ((n % 20) + 1),
            'device', (ARRAY['mobile','desktop'])[(n % 2) + 1])
        WHEN 2 THEN jsonb_build_object(
            'product_id', (n % 200) + 1,
            'price', ((n % 100) + 10)::NUMERIC(10, 2),
            'quantity', (n % 5) + 1,
            'tags', jsonb_build_array((ARRAY['promo','new','clearance'])[(n % 3) + 1]))
        WHEN 3 THEN jsonb_build_object(
            'order_id', 'o_' || (n + 100000),
            'total', ((n % 500) + 50)::NUMERIC(10, 2),
            'items_count', (n % 10) + 1,
            'payment_method', (ARRAY['card','paypal','crypto'])[(n % 3) + 1])
        ELSE jsonb_build_object(
            'model_name', 'rec_v' || ((n % 5) + 1),
            'model_version', '1.' || (n % 10),
            'recommendations', jsonb_build_array((n % 200) + 1, ((n + 1) % 200) + 1, ((n + 2) % 200) + 1),
            'experiment', 'exp_' || ((n % 3) + 1))
    END AS payload
FROM generate_series(1, 50000) AS n;

ANALYZE t7_hw_events;
''')
q("SELECT event_type, COUNT(*) AS n_events, MIN(payload::TEXT) AS sample_payload FROM t7_hw_events GROUP BY event_type ORDER BY event_type")

### JSONB Query 1. Containment `@>`

In [ ]:
q('''
SELECT event_id, user_id, event_ts, payload
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND payload @> jsonb_build_object('payment_method', 'card')
ORDER BY event_id
LIMIT 10
''')

### JSONB Query 2. Key existence `?` та `?|`

In [ ]:
q('''
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ? 'tags'
ORDER BY event_id
LIMIT 5
''')

In [ ]:
q('''
SELECT event_type, COUNT(*) AS n
FROM t7_hw_events
WHERE payload ?| ARRAY['experiment', 'model_version']
GROUP BY event_type
''')

### JSONB Query 3. JSONPath

In [ ]:
q('''
SELECT event_id, jsonb_path_query(payload, '$.recommendations[*]') AS recommended_pid
FROM t7_hw_events
WHERE event_type = 'recommendation_shown'
ORDER BY event_id
LIMIT 15
''')

In [ ]:
q('''
SELECT event_id, payload ->> 'order_id' AS order_id, (payload ->> 'total')::NUMERIC(10, 2) AS total
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND jsonb_path_exists(payload, '$.total ? (@ > 300)')
ORDER BY event_id
LIMIT 10
''')

**JSONB.** 50 000 подій, по 10 000 на кожен з 5 типів, у кожного власна структура payload (у `add_to_cart` — масив `tags`, у `recommendation_shown` — масив `recommendations`). Containment `@>` знаходить checkout з `card` (кожен третій), `?` — лише події з ключем `tags` (усі `add_to_cart`), `?|` з двома ключами — усі 10 000 `recommendation_shown`. JSONPath `$.recommendations[*]` розгортає масив у рядки (три на подію), а `$.total ? (@ > 300)` фільтрує по числовому значенню всередині документа без приведення типів у SQL.

## Завдання 4. GIN-index demo + EXPLAIN ANALYZE

### Step 1. Containment без GIN

In [ ]:
run('''
DROP INDEX IF EXISTS idx_t7_events_payload_gin_pathops;
DROP INDEX IF EXISTS idx_t7_events_payload_gin_ops;
DROP INDEX IF EXISTS idx_t7_events_device_btree;
ANALYZE t7_hw_events;
''')
explain("SELECT COUNT(*) FROM t7_hw_events WHERE payload @> jsonb_build_object('payment_method', 'card')")

### Step 2. GIN-індекс (`jsonb_path_ops`) для containment / JSONPath

In [ ]:
run("CREATE INDEX idx_t7_events_payload_gin_pathops ON t7_hw_events USING GIN (payload jsonb_path_ops); ANALYZE t7_hw_events;")
q('''
SELECT pg_size_pretty(pg_relation_size('idx_t7_events_payload_gin_pathops')) AS gin_index_size,
       pg_size_pretty(pg_relation_size('t7_hw_events'))                     AS table_size
''')

### Step 3. Той самий containment після GIN (два предикати різної селективності)

In [ ]:
explain('''
SELECT COUNT(*) FROM t7_hw_events WHERE payload @> jsonb_build_object('payment_method', 'card')
''')

In [ ]:
explain('''
SELECT COUNT(*) FROM t7_hw_events WHERE payload @> jsonb_build_object('referrer', 'google')
''')

### Step 4. Коли GIN не допомагає: `payload ->> 'device' = 'mobile'` без і з expression B-tree index

In [ ]:
explain('''
SELECT COUNT(*) FROM t7_hw_events WHERE payload ->> 'device' = 'mobile'
''')

In [ ]:
run("CREATE INDEX idx_t7_events_device_btree ON t7_hw_events ((payload ->> 'device')); ANALYZE t7_hw_events;")
explain("SELECT COUNT(*) FROM t7_hw_events WHERE payload ->> 'device' = 'mobile'")

**Ефект індексів за планами.**

| Запит | Без індексу | З індексом |
|---|---|---|
| `payload @> {payment_method: card}` | `Seq Scan`, 13.85 мс, 1 000 буферів, Rows Removed by Filter 46 666 | `Bitmap Index Scan` на GIN + `Bitmap Heap Scan`, **1.99 мс**, 4 буфери індексу + 1 000 heap |
| `payload @> {referrer: google}` | — | `Bitmap Index Scan`, 1.75 мс, 2 500 рядків |
| `payload ->> 'device' = 'mobile'` (є GIN) | `Seq Scan`, 3.90 мс; оцінка планувальника 250 рядків проти фактичних 8 333 | expression B-tree: `Bitmap Index Scan`, **1.38 мс**, оцінка 8 358 ≈ факт |

GIN `jsonb_path_ops` займає 1 144 кБ при таблиці 8 000 кБ (14 %) і прискорив containment у 7 разів: cost 1 758 → 1 107, `Seq Scan` змінився на `Bitmap Index Scan` + `Bitmap Heap Scan`. Але для `->> 'device' = 'mobile'` той самий GIN не використано взагалі — план лишився `Seq Scan`, бо GIN індексує пари ключ/значення для операторів `@>`, `?`, `@?`, а вираз `->>` повертає текст, який порівнюється звичайним `=`; це інший оператор і інший тип даних. Expression B-tree `((payload ->> 'device'))` індексує саме результат виразу, тож для нього план став `Bitmap Index Scan`. Окремо видно, що без expression-індексу планувальник не має статистики по виразу і помиляється в оцінці у 33 рази (250 проти 8 333), а після `CREATE INDEX` + `ANALYZE` з'явилась статистика по виразу і оцінка стала точною.

## Завдання 5. JSONB vs normalized

### Step 1. Нормалізована таблиця для checkout-подій

In [ ]:
run('''
DROP TABLE IF EXISTS t7_hw_checkouts_norm;
CREATE TABLE t7_hw_checkouts_norm (
    event_id        INTEGER PRIMARY KEY,
    user_id         INTEGER NOT NULL,
    event_ts        TIMESTAMPTZ NOT NULL,
    order_id        TEXT NOT NULL,
    total           NUMERIC(10, 2) NOT NULL,
    items_count     INTEGER NOT NULL CHECK (items_count > 0),
    payment_method  TEXT NOT NULL CHECK (payment_method IN ('card', 'paypal', 'crypto'))
);
INSERT INTO t7_hw_checkouts_norm (event_id, user_id, event_ts, order_id, total, items_count, payment_method)
SELECT event_id, user_id, event_ts,
       payload ->> 'order_id',
       (payload ->> 'total')::NUMERIC(10, 2),
       (payload ->> 'items_count')::INTEGER,
       payload ->> 'payment_method'
FROM t7_hw_events
WHERE event_type = 'checkout';
CREATE INDEX idx_t7_checkouts_norm_pm ON t7_hw_checkouts_norm (payment_method);
ANALYZE t7_hw_checkouts_norm;
''')
q('''
SELECT COUNT(*) AS n_checkouts,
       pg_size_pretty(pg_total_relation_size('t7_hw_checkouts_norm')) AS norm_total_size,
       pg_size_pretty(pg_total_relation_size('t7_hw_events')) AS events_total_size
FROM t7_hw_checkouts_norm
''')

### Step 2. Один запит двома способами: середній чек за способом оплати, top-3

In [ ]:
print("--- JSONB-версія ---")
explain('''
SELECT payload ->> 'payment_method' AS payment_method,
       AVG((payload ->> 'total')::NUMERIC) AS avg_total,
       COUNT(*) AS n
FROM t7_hw_events
WHERE event_type = 'checkout'
GROUP BY payload ->> 'payment_method'
ORDER BY avg_total DESC
LIMIT 3
''')
print("\n--- Normalized-версія ---")
explain('''
SELECT payment_method, AVG(total) AS avg_total, COUNT(*) AS n
FROM t7_hw_checkouts_norm
GROUP BY payment_method
ORDER BY avg_total DESC
LIMIT 3
''')

In [ ]:
q('''
-- Результат однаковий в обох версіях
SELECT payment_method, ROUND(AVG(total), 2) AS avg_total, COUNT(*) AS n
FROM t7_hw_checkouts_norm
GROUP BY payment_method
ORDER BY avg_total DESC
LIMIT 3
''')

**JSONB vs normalized.** Результат однаковий (середній чек ≈ 300.5 для всіх трьох способів оплати — синтетика рівномірна), а вартість різна: JSONB-версія 4.36 мс, `Seq Scan` по всій `t7_hw_events` (1 000 буферів, 40 000 рядків відкинуто фільтром `event_type`), `HashAggregate` 217 кБ, бо групується текстовий вираз `payload ->> 'payment_method'` і для кожного рядка парситься документ; normalized — 1.51 мс, `Seq Scan` по компактній таблиці (94 буфери, 1 112 кБ проти 10 МБ), `HashAggregate` 24 кБ по типізованій колонці. У 2.9 раза швидше і в 10 разів менше I/O — за рахунок того, що дані вже розпарсені й типізовані при завантаженні, а не під час кожного запиту. Індекс `payment_method` тут не використано (агрегат по всій таблиці), але він дав би виграш при фільтрі по одному способу оплати.

### Step 3 (bonus). MongoDB-equivalent як pseudocode

```javascript
// MongoDB aggregation pipeline: середній чек за payment_method, top-3
// Pseudocode - не виконується у PostgreSQL
db.events.aggregate([
  { $match: { event_type: "checkout" } },
  { $group: {
      _id:       "$payload.payment_method",
      avg_total: { $avg: { $toDouble: "$payload.total" } },
      n:         { $sum: 1 }
  } },
  { $sort:  { avg_total: -1 } },
  { $limit: 3 }
]);
```

Різниця підходів:
- **PostgreSQL JSONB** доречний, коли поруч із подіями потрібні SQL, JOIN з нормалізованими таблицями (користувачі, замовлення, товари), транзакції та constraints — усе в одній базі, один раз для всього pipeline.
- **MongoDB aggregation pipeline** природний для document-first навантажень, де документ — одиниця роботи, схема змінюється часто, а JOIN рідкісні.
- **Сирий високооб'ємний clickstream** (мільярди подій) краще тримати в document store або data lake (Parquet на S3) з батчевою обробкою, а не в OLTP-PostgreSQL.
- **Curated ML features** зручніше тримати у typed SQL-таблицях: фіксована схема, типи, CHECK, індекси, point-in-time JOIN.

## Reflection

**До і після GIN.** Containment `payload @> {payment_method: card}` без індексу — `Seq Scan`, cost 1 758, 13.85 мс, 46 666 рядків відкинуто фільтром. Після `GIN (payload jsonb_path_ops)` — `Bitmap Index Scan` (4 буфери) + `Bitmap Heap Scan` з `Recheck Cond`, cost 1 107, 1.99 мс: у 7 разів швидше. Для `referrer = google` — 1.75 мс. На 50 тис. рядків планувальник одразу обрав індекс; на малих або неселективних наборах міг би лишити `Seq Scan`.

**Чому GIN не допоміг для `->> 'device' = 'mobile'`.** GIN зберігає хеші пар ключ/значення і обслуговує оператори `@>`, `?`, `@?`; вираз `payload ->> 'device'` повертає `text`, який порівнюється `=` — для цього оператора індексного класу в GIN немає, тому план лишився `Seq Scan` (3.90 мс). Expression B-tree на `((payload ->> 'device'))` індексує обчислений текст, і запит отримав `Bitmap Index Scan` за 1.38 мс, а оцінка рядків виправилась з 250 на 8 358.

**Read/write trade-off.** GIN займає 14 % обсягу таблиці (1 144 кБ) і прискорює читання containment-запитів; дорожчають `INSERT`/`UPDATE` (оновлення індексу з багатьма ключами на документ, pending list, `VACUUM`), а `jsonb_path_ops` не підтримує `?`-оператори — для них потрібен важчий `jsonb_ops`.

**JSONB чи normalized.** JSONB — для сирого event-log з різнорідними payload, де схема змінюється і потрібен один стіл для всіх типів; normalized — для curated ML-ознак і частих агрегацій: той самий запит виконався за 1.51 мс проти 4.36 мс при 94 буферах проти 1 000.

**Новий event-type `refund`.** У JSONB — просто `INSERT` з новим `payload` без міграції; ризик — тиха зміна схеми, тому потрібен `CHECK (event_type IN (...))` або реєстр схем. У normalized — `CREATE TABLE t7_hw_refunds_norm`, міграція з обмеженнями, FK на checkout, оновлення ETL і backfill історичних подій.